# Modelado Predictivo vs Explicativo
## Censo General de Población y Vivienda 2000 — INEGI
### Programación II — Maestría en Ciencia de Datos | Universidad de Guadalajara CUCEA

---

## ¿Por qué comparar dos enfoques?

En ciencia de datos existe una tensión fundamental entre dos objetivos que no siempre van juntos: **entender** y **predecir**.

El **modelado explicativo** busca cuantificar relaciones entre variables. La pregunta es *¿cuánto y en qué dirección afecta el nivel educativo la probabilidad de tener cobertura médica?* Se usan modelos estadísticos con supuestos paramétricos claros (Logit, OLS), donde cada coeficiente tiene una interpretación directa y una prueba de significancia estadística. El objetivo no es predecir bien, sino entender el mecanismo.

El **modelado predictivo** busca maximizar la precisión sobre datos nuevos. No le importa tanto *por qué* ocurre algo, sino *con qué precisión* puede anticiparlo. Se usan algoritmos de machine learning (Decision Tree, Random Forest) que capturan relaciones no lineales que un modelo paramétrico no puede ver, pero a costa de perder interpretabilidad directa.

La diferencia no es técnica, es epistemológica:

| | Explicativo | Predictivo |
|---|---|---|
| Pregunta | ¿Por qué ocurre? | ¿Qué tan bien lo predigo? |
| Modelo | Logit, OLS | Decision Tree, Random Forest |
| Salida clave | Coeficientes + p-valores | Accuracy, R², RMSE |
| Limitación | Supuestos paramétricos | Caja negra |
| Ideal para | Investigación, política pública | Sistemas en producción |

---

**Objetivo:** Aplicar ambos enfoques a dos preguntas de investigación sobre el Censo INEGI 2000 y comparar qué nos dice cada uno sobre el mismo fenómeno.

- **Modelo 1:** ¿Qué factores determinan el acceso a derechohabiencia?
- **Modelo 2:** ¿Qué factores determinan la edad de muerte?

**Pipeline:**
```
SQL Server (10M registros) → Chunks 100K → Preprocesamiento → Logit + Decision Tree + OLS + Random Forest → Comparativa
```

## 0. Configuración e importaciones

In [1]:
import os
import warnings
import matplotlib

# Headless en Linux sin DISPLAY (nbconvert); en Windows/Jupyter usa el backend por defecto
if os.name != 'nt' and not os.environ.get('DISPLAY'):
    matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyodbc
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_squared_error

# Statsmodels
import statsmodels.api as sm

from pathlib import Path

# Rutas — FIGURES_DIR es relativo al repo, no al notebook
FIGURES_DIR = Path.cwd().parent / 'figures' if Path.cwd().name == 'notebooks' else Path.cwd() / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

print('Librerias importadas correctamente')
print(f'Backend matplotlib : {matplotlib.get_backend()}')
print(f'Imagenes se guardaran en: {FIGURES_DIR.resolve()}')

Librerias importadas correctamente
Backend matplotlib : Agg
Imagenes se guardaran en: /home/carpuro/mcd/censo-salud-mexico-2000/figures


In [2]:
# Conexion a SQL Server
REPO_DIR = FIGURES_DIR.parent
load_dotenv(dotenv_path=REPO_DIR / '.env')

DB_SERVER = os.getenv('DB_SERVER', 'localhost')
DB_PORT   = os.getenv('DB_PORT', '1433')
DB_NAME   = os.getenv('DB_NAME', 'Defunciones_2000')
DB_USER   = os.getenv('DB_USER')
DB_PASS   = os.getenv('DB_PASSWORD')

# Intentar Driver 18 primero, luego Driver 17 (compatibilidad Windows)
conn = None
for driver in ['ODBC Driver 18 for SQL Server', 'ODBC Driver 17 for SQL Server']:
    try:
        conn = pyodbc.connect(
            f'DRIVER={{{driver}}};'
            f'SERVER={DB_SERVER},{DB_PORT};'
            f'DATABASE={DB_NAME};'
            f'UID={DB_USER};'
            f'PWD={DB_PASS};'
            f'TrustServerCertificate=yes;'
        )
        print(f'Conexion exitosa usando: {driver}')
        break
    except pyodbc.Error:
        print(f'Driver no disponible: {driver}')

if conn is None:
    raise RuntimeError('No se pudo conectar. Instalar ODBC Driver 17 o 18 for SQL Server.')

print(f'Base de datos: {DB_NAME} en {DB_SERVER}')

Conexion exitosa usando: ODBC Driver 18 for SQL Server
Base de datos: Defunciones_2000 en localhost


## 1. Carga de datos por chunks

Dado el volumen de 10 millones de registros, cargamos los datos en bloques de 100,000 registros para un manejo eficiente de memoria.

In [3]:
query = '''
    SELECT 
        CAST(ENT AS INT)       AS ENT,
        CAST(SEXO AS INT)      AS SEXO,
        CAST(EDAD AS INT)      AS EDAD,
        CAST(IMSS AS INT)      AS IMSS,
        CAST(ISSSTE AS INT)    AS ISSSTE,
        CAST(NOTIEDER AS INT)  AS NOTIEDER,
        CAST(NIVELACAD AS INT) AS NIVELACAD,
        CAST(TAM_LOC AS INT)   AS TAM_LOC
    FROM defunciones
    WHERE EDAD NOT IN (\'999\', \'\')
      AND SEXO IN (\'1\', \'2\')
      AND ENT IS NOT NULL
      AND EDAD IS NOT NULL
'''

chunks = []
total = 0

for chunk in pd.read_sql(query, conn, chunksize=100000):
    chunks.append(chunk)
    total += len(chunk)
    print(f'Registros cargados: {total:,}')

df = pd.concat(chunks, ignore_index=True)
print(f'\nCarga completa: {len(df):,} registros totales')

Registros cargados: 100,000


Registros cargados: 200,000


Registros cargados: 300,000


Registros cargados: 400,000
Registros cargados: 500,000


Registros cargados: 600,000


...

Registros cargados: 9,400,000
Registros cargados: 9,500,000


Registros cargados: 9,600,000
Registros cargados: 9,700,000


Registros cargados: 9,800,000


Registros cargados: 9,900,000
Registros cargados: 10,000,000


Registros cargados: 10,065,492

Carga completa: 10,065,492 registros totales


## 2. Preprocesamiento y definición de variables

### Variables independientes (features)
| Variable | Descripción | Tipo |
|----------|-------------|------|
| ENT | Estado de residencia (1-32) | Categórica |
| SEXO | Sexo (1=Hombre, 2=Mujer) | Categórica |
| EDAD | Edad en años (0-120) | Numérica |
| NIVELACAD | Nivel académico (1=Sin escolaridad... 8=Posgrado) | Ordinal |
| TAM_LOC | Tamaño localidad (1=Rural... 4=Metropolitano) | Ordinal |

### Variables dependientes
| Variable | Descripción | Modelo |
|----------|-------------|--------|
| tiene_derechohabiencia | 1=tiene cobertura, 0=no tiene | Modelo 1 |
| EDAD | Edad de muerte en años | Modelo 2 |

In [4]:
# Limpiar y preparar datos
df = df.apply(pd.to_numeric, errors='coerce')

# Crear variable target ANTES de dropna
# NOTIEDER puede ser NULL en la BD (NULL = con cobertura; 5 = sin cobertura)
# np.where: NaN == 5 es False en pandas, por lo tanto NaN -> 1 (con cobertura)
df['tiene_derechohabiencia'] = np.where(df['NOTIEDER'] == 5, 0, 1)

# Filtrar solo en columnas de features — NOTIEDER NO es feature, es fuente del target
columnas_features = ['ENT', 'SEXO', 'EDAD', 'NIVELACAD', 'TAM_LOC']
df = df.dropna(subset=columnas_features)

# Filtrar edades validas (0-120)
df = df[(df['EDAD'] >= 0) & (df['EDAD'] <= 120)]

# Dataset adultos para Modelo 2 — excluir mortalidad infantil
df_adultos = df[df['EDAD'] >= 15].copy()

print(f'Dataset completo : {len(df):,} registros')
print(f'Dataset adultos  : {len(df_adultos):,} registros')
print(f'\n--- Modelo 1: Derechohabiencia ---')
print(f'Con cobertura    : {df["tiene_derechohabiencia"].sum():,} ({df["tiene_derechohabiencia"].mean()*100:.1f}%)')
print(f'Sin cobertura    : {(df["tiene_derechohabiencia"]==0).sum():,} ({(1-df["tiene_derechohabiencia"].mean())*100:.1f}%)')
print(f'\n--- Modelo 2: Edad de Muerte ---')
print(f'Edad promedio    : {df_adultos["EDAD"].mean():.2f} anos')
print(f'Edad minima      : {df_adultos["EDAD"].min():.0f} anos')
print(f'Edad maxima      : {df_adultos["EDAD"].max():.0f} anos')
print(f'Desv. estandar   : {df_adultos["EDAD"].std():.2f} anos')

Dataset completo : 8,896,028 registros
Dataset adultos  : 6,451,966 registros

--- Modelo 1: Derechohabiencia ---
Con cobertura    : 3,087,862 (34.7%)
Sin cobertura    : 5,808,166 (65.3%)

--- Modelo 2: Edad de Muerte ---
Edad promedio    : 36.43 anos
Edad minima      : 15 anos
Edad maxima      : 120 anos
Desv. estandar   : 17.10 anos


In [5]:
# Visualizacion de variables dependientes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Variables Dependientes — Censo 2000 INEGI', fontsize=14, fontweight='bold')

# Modelo 1: Derechohabiencia — bar chart (mas estable que pie)
sin_cob = int((df['tiene_derechohabiencia'] == 0).sum())
con_cob = int((df['tiene_derechohabiencia'] == 1).sum())
total   = sin_cob + con_cob
categorias = ['Sin derechohabiencia', 'Con derechohabiencia']
valores    = [sin_cob / total * 100, con_cob / total * 100]
colores    = ['#F44336', '#2196F3']
bars = axes[0].bar(categorias, valores, color=colores, width=0.5)
axes[0].set_title('Modelo 1: Distribucion de Derechohabiencia', fontsize=11)
axes[0].set_ylabel('Porcentaje (%)')
axes[0].set_ylim(0, 100)
for bar, val in zip(bars, valores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', fontweight='bold')

# Modelo 2: Distribucion de edades
axes[1].hist(df_adultos['EDAD'], bins=50, color='#4CAF50', edgecolor='white', alpha=0.8)
axes[1].axvline(df_adultos['EDAD'].mean(), color='red', linestyle='--',
                label=f'Media: {df_adultos["EDAD"].mean():.1f} anos')
axes[1].axvline(df_adultos['EDAD'].median(), color='orange', linestyle='--',
                label=f'Mediana: {df_adultos["EDAD"].median():.1f} anos')
axes[1].set_title('Modelo 2: Distribucion de Edad de Muerte', fontsize=11)
axes[1].set_xlabel('Edad')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
ruta = FIGURES_DIR / 'variables_dependientes.png'
plt.savefig(ruta, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {ruta}')

Guardado: /home/carpuro/mcd/censo-salud-mexico-2000/figures/variables_dependientes.png


---
## Análisis Descriptivo — Panorama General
---

In [ ]:
# Dashboard descriptivo — datos desde tablas materializadas (rapido)
import matplotlib.gridspec as gridspec

kpis        = pd.read_sql('SELECT * FROM tbl_kpis', conn)
estados_db  = pd.read_sql('SELECT * FROM tbl_estados_resumen ORDER BY pct_sin_cobertura DESC', conn)
urbano_db   = pd.read_sql('SELECT * FROM tbl_urbano_resumen WHERE tipo_localidad_codigo != 7 ORDER BY tipo_localidad_codigo', conn)
escolar_db  = pd.read_sql('SELECT * FROM tbl_escolaridad_resumen ORDER BY orden', conn)

C_SIN  = '#E53935'
C_IMSS = '#1E88E5'
C_ISS  = '#43A047'
C_PEM  = '#FB8C00'
C_OTRO = '#9E9E9E'

fig = plt.figure(figsize=(16, 11))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('Acceso a Servicios de Salud en Mexico — Censo 2000',
             fontsize=16, fontweight='bold', y=0.98)

gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.32)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

# Pie — distribucion general
total   = kpis['total_poblacion'].iloc[0]
imss    = kpis['total_con_imss'].iloc[0]
issste  = kpis['total_con_issste'].iloc[0]
pemex   = kpis['total_con_pemex'].iloc[0]
sin_cob = kpis['total_sin_derechohabiencia'].iloc[0]
otro    = total - imss - issste - pemex - sin_cob

sizes   = [sin_cob, imss, issste, pemex, otro]
labels  = ['Sin derechohabiencia', 'IMSS', 'ISSSTE', 'PEMEX/SEDENA', 'Otro']
colors  = [C_SIN, C_IMSS, C_ISS, C_PEM, C_OTRO]
wedges, texts, autotexts = ax1.pie(
    sizes, labels=None, colors=colors, autopct='%1.1f%%',
    startangle=140, explode=[0.04,0,0,0,0],
    textprops={'fontsize': 9}, pctdistance=0.82
)
for at, c in zip(autotexts, colors):
    at.set_color('white' if c != C_OTRO else '#333')
    at.set_fontweight('bold')
ax1.legend(wedges, labels, loc='lower center', bbox_to_anchor=(0.5, -0.18),
           ncol=2, fontsize=8, frameon=False)
ax1.set_title('Distribucion de Acceso a Salud', fontsize=11, fontweight='bold', pad=10)
ax1.text(0, -1.55,
    f'Solo el {100-kpis["pct_sin_derechohabiencia"].iloc[0]:.1f}% tenia cobertura formal.\n'
    f'ISSSTE: {issste/1e6:.2f}M personas  |  PEMEX/SEDENA: {pemex/1e6:.2f}M personas',
    ha='center', fontsize=7.5, color='#555', transform=ax1.transData)

# Top 10 estados
top10 = estados_db.head(10).sort_values('pct_sin_cobertura')
bars  = ax2.barh(top10['nombre_estado'], top10['pct_sin_cobertura'], color=C_SIN, alpha=0.85)
ax2.set_xlabel('% sin derechohabiencia', fontsize=9)
ax2.set_title('Top 10 Estados Sin Derechohabiencia', fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', labelsize=8)
ax2.set_xlim(0, 100)
prom = kpis['pct_sin_derechohabiencia'].iloc[0]
ax2.axvline(prom, color='#333', linestyle='--', linewidth=1, alpha=0.6,
            label=f'Promedio nacional ({prom:.1f}%)')
ax2.legend(fontsize=7.5, frameon=False)
for bar, val in zip(bars, top10['pct_sin_cobertura']):
    ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=7.5)
ax2.text(0.5, -0.16,
    'Los estados con mayor poblacion rural e indigena concentran\nlos niveles mas altos de poblacion sin cobertura.',
    ha='center', fontsize=7.5, color='#555', transform=ax2.transAxes)

# Urbano vs rural
labels_urb = [t.split('(')[0].strip() for t in urbano_db['tipo_localidad']]
x  = np.arange(len(labels_urb))
w  = 0.35
ax3.bar(x - w/2, urbano_db['con_imss']/1e6,              w, label='Con IMSS',     color=C_IMSS, alpha=0.85)
ax3.bar(x + w/2, urbano_db['sin_derechohabiencia']/1e6,  w, label='Sin cobertura',color=C_SIN,  alpha=0.85)
ax3.set_xticks(x)
ax3.set_xticklabels(labels_urb, rotation=20, ha='right', fontsize=8)
ax3.set_ylabel('Millones de personas', fontsize=9)
ax3.set_title('Cobertura Urbano vs Rural', fontsize=11, fontweight='bold')
ax3.legend(fontsize=8.5, frameon=False)
ax3.text(0.5, -0.26,
    'La brecha entre zonas rurales y megalopolis es la mas pronunciada.\n'
    'En zonas rurales el 83.7% de la poblacion carece de cobertura medica.',
    ha='center', fontsize=7.5, color='#555', transform=ax3.transAxes)

# Nivel educativo
x4 = np.arange(len(escolar_db))
ax4.barh(x4 - 0.2, escolar_db['con_imss']/1e3,             0.35, label='Con IMSS',     color=C_IMSS, alpha=0.85)
ax4.barh(x4 + 0.2, escolar_db['sin_derechohabiencia']/1e3, 0.35, label='Sin cobertura',color=C_SIN,  alpha=0.85)
ax4.set_yticks(x4)
ax4.set_yticklabels(escolar_db['nivel_educativo'], fontsize=8)
ax4.set_xlabel('Miles de personas', fontsize=9)
ax4.set_title('Cobertura por Nivel Educativo', fontsize=11, fontweight='bold')
ax4.legend(fontsize=8.5, frameon=False)
ax4.text(0.5, -0.16,
    'La educacion actua como factor protector.\n'
    'Licenciatura y posgrado son los unicos niveles donde la cobertura supera la desproteccion.',
    ha='center', fontsize=7.5, color='#555', transform=ax4.transAxes)

plt.tight_layout()
ruta = FIGURES_DIR / 'narrativa_salud_2000.png'
plt.savefig(ruta, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Guardado: {ruta}')

---
# MODELO 1: Derechohabiencia
### ¿Qué factores determinan el acceso a servicios de salud?
---

## 3A. Modelo 1 — Enfoque EXPLICATIVO (Logit)

La regresión logística nos permite cuantificar **cuánto y en qué dirección** cada variable afecta la *probabilidad* de tener derechohabiencia. Los coeficientes son directamente interpretables: un coeficiente positivo significa que esa variable aumenta la probabilidad de tener cobertura.

Usamos **efectos marginales** (`get_margeff()`) en lugar de los coeficientes crudos del Logit, porque los coeficientes del Logit están en escala log-odds — difíciles de interpretar directamente. Los efectos marginales traducen eso a cambios en probabilidad: *"cada nivel adicional de educación aumenta la probabilidad de tener cobertura en X puntos porcentuales"*.

**Muestra de 50,000 registros** — statsmodels con toda la tabla (8.9M) tardaría varios minutos; la muestra aleatoria es estadísticamente representativa para inferencia.

In [6]:
# Muestra para modelo explicativo (statsmodels es lento con millones)
df_sample = df.sample(n=50000, random_state=42)

X_exp = df_sample[['SEXO', 'EDAD', 'NIVELACAD', 'TAM_LOC', 'ENT']].copy()
y_exp = df_sample['tiene_derechohabiencia']

X_exp_const = sm.add_constant(X_exp)

print('Ajustando modelo Logit...')
modelo_logit = sm.Logit(y_exp, X_exp_const)
resultado_logit = modelo_logit.fit(disp=False)

print('\nModelo ajustado')
print(resultado_logit.summary())

Ajustando modelo Logit...

Modelo ajustado
                             Logit Regression Results                             
Dep. Variable:     tiene_derechohabiencia   No. Observations:                50000
Model:                              Logit   Df Residuals:                    49994
Method:                               MLE   Df Model:                            5
Date:                    Wed, 01 Apr 2026   Pseudo R-squ.:                  0.1224
Time:                            11:13:28   Log-Likelihood:                -28350.
converged:                           True   LL-Null:                       -32305.
Covariance Type:                nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -2.3649      0.049    -48.273      0.000      -2.461      -2.269
SEXO           0.0472      0.020      2.308      0.021  

In [7]:
# Efectos marginales
efectos = resultado_logit.get_margeff()
print('--- Efectos Marginales (impacto real en probabilidad) ---')
print(efectos.summary())

print('\n--- Interpretacion ---')
coefs = resultado_logit.params
pvals = resultado_logit.pvalues

variables = {
    'SEXO'     : 'Ser mujer vs hombre',
    'EDAD'     : 'Cada anio adicional de edad',
    'NIVELACAD': 'Cada nivel educativo adicional',
    'TAM_LOC'  : 'Cada nivel de urbanizacion',
    'ENT'      : 'Estado de residencia'
}

for var, desc in variables.items():
    sig = 'Significativo' if pvals[var] < 0.05 else 'No significativo'
    direccion = 'Aumenta' if coefs[var] > 0 else 'Reduce'
    print(f'{desc:<35}: {direccion} probabilidad | {sig} (p={pvals[var]:.4f})')

--- Efectos Marginales (impacto real en probabilidad) ---
          Logit Marginal Effects         
Dep. Variable:     tiene_derechohabiencia
Method:                              dydx
At:                               overall
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
SEXO           0.0090      0.004      2.308      0.021       0.001       0.017
EDAD           0.0013      0.000     12.756      0.000       0.001       0.002
NIVELACAD      0.0024   9.63e-05     25.346      0.000       0.002       0.003
TAM_LOC        0.0591      0.001     88.718      0.000       0.058       0.060
ENT            0.0002      0.000      0.684      0.494      -0.000       0.001

--- Interpretacion ---
Ser mujer vs hombre                : Aumenta probabilidad | Significativo (p=0.0210)
Cada anio adicional de edad        : Aumenta probabilidad | Significativo (p=0.0000)
Cada nivel educativo adicio

## 3B. Modelo 1 — Enfoque PREDICTIVO (Decision Tree)

El árbol de decisión hace una pregunta completamente distinta: dado el perfil demográfico de una persona, ¿podemos *predecir* correctamente si tiene o no derechohabiencia?

A diferencia del Logit, el árbol no asume ninguna forma funcional ni linealidad. Parte los datos de forma recursiva buscando el split que mejor separe las clases. El resultado es una secuencia de reglas del tipo *"si NIVELACAD > 3 y TAM_LOC > 2, entonces probablemente tiene cobertura"*.

Lo que el árbol **no puede hacer**: decirte si esa relación es causal, cuál es el intervalo de confianza, o si el efecto es estadísticamente significativo. Para eso está el Logit.

**Parámetros:** `max_depth=5` — limita la profundidad para evitar sobreajuste; con datos de censo la estructura es relativamente simple.

In [8]:
X_pred = df[['SEXO', 'EDAD', 'NIVELACAD', 'TAM_LOC', 'ENT']]
y_pred = df['tiene_derechohabiencia']

X_train, X_test, y_train, y_test = train_test_split(
    X_pred, y_pred, test_size=0.2, random_state=42
)

print(f'Entrenamiento : {len(X_train):,} registros')
print(f'Prueba        : {len(X_test):,} registros')

print('\nEntrenando Decision Tree...')
modelo_dt = DecisionTreeClassifier(max_depth=5, random_state=42)
modelo_dt.fit(X_train, y_train)

predicciones_dt = modelo_dt.predict(X_test)
accuracy = accuracy_score(y_test, predicciones_dt)

print(f'\nModelo entrenado')
print(f'Accuracy: {accuracy*100:.2f}%')
print(f'\n--- Reporte de Clasificacion ---')
print(classification_report(
    y_test, predicciones_dt,
    target_names=['Sin derechohabiencia', 'Con derechohabiencia']
))

Entrenamiento : 7,116,822 registros
Prueba        : 1,779,206 registros

Entrenando Decision Tree...



Modelo entrenado
Accuracy: 71.45%

--- Reporte de Clasificacion ---


                      precision    recall  f1-score   support

Sin derechohabiencia       0.73      0.89      0.80   1160472
Con derechohabiencia       0.65      0.38      0.48    618734

            accuracy                           0.71   1779206
           macro avg       0.69      0.64      0.64   1779206
        weighted avg       0.70      0.71      0.69   1779206



In [9]:
# Importancia de variables — Modelo 1
importancias_m1 = pd.DataFrame({
    'Variable'   : ['Estado', 'Sexo', 'Edad', 'Nivel Educativo', 'Tipo Localidad'],
    'Importancia': modelo_dt.feature_importances_
}).sort_values('Importancia', ascending=True)

print('--- Importancia de Variables ---')
print(importancias_m1.to_string(index=False))

plt.figure(figsize=(8, 4))
colores = ['#FF9800', '#9C27B0', '#2196F3', '#4CAF50', '#F44336']
plt.barh(importancias_m1['Variable'], importancias_m1['Importancia'], color=colores)
plt.title('Importancia de Variables — Prediccion Derechohabiencia', fontsize=12)
plt.xlabel('Importancia relativa')
for i, v in enumerate(importancias_m1['Importancia']):
    plt.text(v + 0.001, i, f'{v:.3f}', va='center')
plt.tight_layout()
ruta = FIGURES_DIR / 'importancia_modelo1.png'
plt.savefig(ruta, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {ruta}')

--- Importancia de Variables ---
       Variable  Importancia
         Estado     0.000000
           Sexo     0.025566
 Tipo Localidad     0.041600
           Edad     0.161867
Nivel Educativo     0.770967
Guardado: /home/carpuro/mcd/censo-salud-mexico-2000/figures/importancia_modelo1.png


## 3C. Comparativa Modelo 1 — ¿Qué nos dice cada enfoque?

El Logit y el Decision Tree analizan los mismos datos pero responden preguntas distintas. Antes de ver los números, vale la pena enmarcar qué significa cada métrica:

- **Pseudo R² (Logit):** no es un R² tradicional. Mide cuánto mejora el modelo respecto a predecir siempre la clase mayoritaria. Un valor de 0.12 es razonable para datos sociodemográficos con alta varianza individual.
- **Accuracy (Decision Tree):** porcentaje de casos clasificados correctamente. Más intuitivo, pero sensible al desbalance de clases.

La coincidencia más relevante entre ambos enfoques es cuáles variables identifican como importantes. Si el Logit dice que la educación es significativa Y el árbol le asigna alta importancia, eso es evidencia convergente desde dos metodologías distintas.

In [10]:
print('=' * 60)
print('COMPARATIVA MODELO 1 — DERECHOHABIENCIA')
print('=' * 60)

print('\nENFOQUE EXPLICATIVO (statsmodels Logit)')
print('-' * 40)
print(f'Pseudo R2         : {resultado_logit.prsquared:.4f}')
print(f'Log-Likelihood    : {resultado_logit.llf:.2f}')
print(f'AIC               : {resultado_logit.aic:.2f}')
print('Pregunta          : Por que alguien tiene derechohabiencia?')

print('\nENFOQUE PREDICTIVO (Decision Tree)')
print('-' * 40)
print(f'Accuracy          : {accuracy*100:.2f}%')
print(f'Variable mas imp. : {importancias_m1.iloc[-1]["Variable"]} ({importancias_m1.iloc[-1]["Importancia"]:.3f})')
print('Pregunta          : Tendra derechohabiencia esta persona?')

print('\nCONCLUSION')
print('-' * 40)
print('Explicativo: el POR QUE y CUANTO')
print('Predictivo : QUE TAN BIEN podemos predecir')
print('Ambos confirman que estado y educacion son los')
print('factores mas determinantes del acceso a salud')

COMPARATIVA MODELO 1 — DERECHOHABIENCIA

ENFOQUE EXPLICATIVO (statsmodels Logit)
----------------------------------------
Pseudo R2         : 0.1224
Log-Likelihood    : -28350.06
AIC               : 56712.12
Pregunta          : Por que alguien tiene derechohabiencia?

ENFOQUE PREDICTIVO (Decision Tree)
----------------------------------------
Accuracy          : 71.45%
Variable mas imp. : Nivel Educativo (0.771)
Pregunta          : Tendra derechohabiencia esta persona?

CONCLUSION
----------------------------------------
Explicativo: el POR QUE y CUANTO
Predictivo : QUE TAN BIEN podemos predecir
Ambos confirman que estado y educacion son los
factores mas determinantes del acceso a salud


---
# MODELO 2: Edad de Muerte
### ¿Qué factores determinan a qué edad muere una persona?
---

## 4A. Modelo 2 — Enfoque EXPLICATIVO (OLS)

La regresión OLS cuantifica cuántos **años de vida adicionales** (o menos) están asociados a cada variable demográfica. La interpretación es directa: si el coeficiente de `tiene_derechohabiencia` es +3.5, significa que en promedio las personas con cobertura médica murieron 3.5 años más tarde que las sin cobertura, controlando por las demás variables.

Dos cosas importantes sobre el R² esperado:
1. La edad de muerte tiene altísima varianza individual — causas genéticas, accidentes, enfermedades específicas no están en los datos
2. Variables sociodemográficas capturan contexto estructural, no biología individual

Un R² bajo no invalida el modelo explicativo. Lo que importa es si los coeficientes son estadísticamente significativos y tienen sentido teórico.

**Muestra de 50,000 adultos (≥15 años)** — mismo criterio que el Logit para comparabilidad.

In [11]:
df_sample2 = df_adultos.sample(n=50000, random_state=42)

X_exp2 = df_sample2[['SEXO', 'NIVELACAD', 'TAM_LOC', 'ENT', 'tiene_derechohabiencia']].copy()
y_exp2 = df_sample2['EDAD']

X_exp2_const = sm.add_constant(X_exp2)

print('Ajustando modelo OLS...')
modelo_ols = sm.OLS(y_exp2, X_exp2_const)
resultado_ols = modelo_ols.fit()

print('\nModelo ajustado')
print(resultado_ols.summary())

Ajustando modelo OLS...

Modelo ajustado
                            OLS Regression Results                            
Dep. Variable:                   EDAD   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.028
Method:                 Least Squares   F-statistic:                     287.5
Date:                Wed, 01 Apr 2026   Prob (F-statistic):          2.52e-304
Time:                        11:13:35   Log-Likelihood:            -2.1198e+05
No. Observations:               50000   AIC:                         4.240e+05
Df Residuals:                   49994   BIC:                         4.240e+05
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------

In [12]:
print('--- Interpretacion de Coeficientes OLS ---')
print('(cada coeficiente = anios de vida mas o menos)\n')

coefs_ols = resultado_ols.params
pvals_ols = resultado_ols.pvalues

variables2 = {
    'SEXO'                   : 'Ser mujer vs hombre',
    'NIVELACAD'              : 'Cada nivel educativo adicional',
    'TAM_LOC'                : 'Cada nivel de urbanizacion',
    'ENT'                    : 'Estado de residencia',
    'tiene_derechohabiencia' : 'Tener derechohabiencia'
}

for var, desc in variables2.items():
    sig = 'Significativo' if pvals_ols[var] < 0.05 else 'No significativo'
    anios = coefs_ols[var]
    direccion = f'+{anios:.2f} anios' if anios > 0 else f'{anios:.2f} anios'
    print(f'{desc:<35}: {direccion} | {sig} (p={pvals_ols[var]:.4f})')

print(f'\nR2 del modelo  : {resultado_ols.rsquared:.4f}')
print(f'R2 ajustado    : {resultado_ols.rsquared_adj:.4f}')

--- Interpretacion de Coeficientes OLS ---
(cada coeficiente = anios de vida mas o menos)

Ser mujer vs hombre                : -0.01 anios | No significativo (p=0.9612)
Cada nivel educativo adicional     : -0.12 anios | Significativo (p=0.0000)
Cada nivel de urbanizacion         : -0.14 anios | Significativo (p=0.0001)
Estado de residencia               : +0.02 anios | No significativo (p=0.0661)
Tener derechohabiencia             : +3.46 anios | Significativo (p=0.0000)

R2 del modelo  : 0.0279
R2 ajustado    : 0.0279


## 4B. Modelo 2 — Enfoque PREDICTIVO (Random Forest Regressor)

El Random Forest construye 100 árboles de decisión sobre muestras aleatorias del dataset y promedia sus predicciones. Al combinar muchos árboles con subconjuntos distintos de datos y variables, reduce el sobreajuste que tendría un árbol individual y captura interacciones no lineales entre variables.

Para este problema de regresión (predecir una edad numérica), usamos R² y RMSE:
- **R²:** qué proporción de la varianza de la edad de muerte explica el modelo
- **RMSE:** error promedio en años — más interpretable que R² para comunicar resultados

La comparación clave con OLS es directa: ambos usan las mismas variables y el mismo split train/test. Si el Random Forest obtiene R² considerablemente mayor, confirma que existen relaciones no lineales que el OLS no puede capturar.

**Parámetros:** 100 árboles, `max_depth=10`, `n_jobs=-1` (todos los núcleos disponibles).

In [13]:
X_pred2 = df_adultos[['SEXO', 'NIVELACAD', 'TAM_LOC', 'ENT', 'tiene_derechohabiencia']]
y_pred2 = df_adultos['EDAD']

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_pred2, y_pred2, test_size=0.2, random_state=42
)

print(f'Entrenamiento : {len(X_train2):,} registros')
print(f'Prueba        : {len(X_test2):,} registros')

print('\nEntrenando Random Forest Regressor (100 arboles, profundidad 10)...')
modelo_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
modelo_rf.fit(X_train2, y_train2)

predicciones_rf = modelo_rf.predict(X_test2)
r2_rf   = r2_score(y_test2, predicciones_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test2, predicciones_rf))

print(f'\nModelo entrenado')
print(f'R2   : {r2_rf:.4f}')
print(f'RMSE : {rmse_rf:.2f} anios')

Entrenamiento : 5,161,572 registros
Prueba        : 1,290,394 registros

Entrenando Random Forest Regressor (100 arboles, profundidad 10)...



Modelo entrenado
R2   : 0.2696
RMSE : 14.61 anios


In [14]:
# Importancia de variables — Modelo 2
importancias_m2 = pd.DataFrame({
    'Variable'   : ['Sexo', 'Nivel Educativo', 'Tipo Localidad', 'Estado', 'Derechohabiencia'],
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=True)

print('--- Importancia de Variables (Random Forest) ---')
print(importancias_m2.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Modelo 2 — Prediccion de Edad de Muerte', fontsize=13, fontweight='bold')

colores2 = ['#FF9800', '#9C27B0', '#2196F3', '#4CAF50', '#F44336']
axes[0].barh(importancias_m2['Variable'], importancias_m2['Importancia'], color=colores2)
axes[0].set_title('Importancia de Variables', fontsize=11)
axes[0].set_xlabel('Importancia relativa')
for i, v in enumerate(importancias_m2['Importancia']):
    axes[0].text(v + 0.001, i, f'{v:.3f}', va='center')

idx = np.random.choice(len(y_test2), 1000, replace=False)
axes[1].scatter(
    np.array(y_test2)[idx],
    predicciones_rf[idx],
    alpha=0.3, color='#2196F3', s=10
)
axes[1].plot([15, 120], [15, 120], 'r--', label='Prediccion perfecta')
axes[1].set_title(f'Predicho vs Real (R2={r2_rf:.3f})', fontsize=11)
axes[1].set_xlabel('Edad real')
axes[1].set_ylabel('Edad predicha')
axes[1].legend()

plt.tight_layout()
ruta = FIGURES_DIR / 'modelo2_resultados.png'
plt.savefig(ruta, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {ruta}')

--- Importancia de Variables (Random Forest) ---
        Variable  Importancia
            Sexo     0.009564
  Tipo Localidad     0.029851
          Estado     0.054530
Derechohabiencia     0.092289
 Nivel Educativo     0.813766


Guardado: /home/carpuro/mcd/censo-salud-mexico-2000/figures/modelo2_resultados.png


## 4C. Comparativa Modelo 2 — ¿Qué aprendemos de cada enfoque?

El OLS y el Random Forest revelan dimensiones distintas del mismo fenómeno:

El **OLS** es valioso aquí no por su R² (que será bajo), sino porque cuantifica el **efecto de la derechohabiencia en años de vida** con significancia estadística. Eso tiene valor de política pública: podemos decir con precisión estadística cuántos años de vida adicionales está asociado tener cobertura médica en México en el año 2000.

El **Random Forest** captura las relaciones no lineales que el OLS no puede ver. Si su R² es sustancialmente mayor, no significa que el OLS "está mal" — significa que la relación entre variables demográficas y edad de muerte es más compleja que lo que una línea recta puede representar.

Ambos enfoques son complementarios: el OLS explica *cuánto*, el Random Forest explica *con qué precisión podemos anticiparlo*.

In [15]:
print('=' * 60)
print('COMPARATIVA MODELO 2 — EDAD DE MUERTE')
print('=' * 60)

print('\nENFOQUE EXPLICATIVO (statsmodels OLS)')
print('-' * 40)
print(f'R2              : {resultado_ols.rsquared:.4f}')
print(f'R2 ajustado     : {resultado_ols.rsquared_adj:.4f}')
print(f'AIC             : {resultado_ols.aic:.2f}')
coef_derecho = resultado_ols.params['tiene_derechohabiencia']
print(f'Efecto derechohabiencia: {coef_derecho:+.2f} anios de vida')
print('Pregunta        : Cuantos anios mas vive quien tiene cobertura?')

print('\nENFOQUE PREDICTIVO (Random Forest)')
print('-' * 40)
print(f'R2              : {r2_rf:.4f}')
print(f'RMSE            : {rmse_rf:.2f} anios')
print(f'Variable mas imp: {importancias_m2.iloc[-1]["Variable"]} ({importancias_m2.iloc[-1]["Importancia"]:.3f})')
print('Pregunta        : A que edad morira esta persona?')

print('\nCONCLUSION')
print('-' * 40)
print('OLS: cuantos anios de vida APORTA cada variable')
print('Random Forest: con que PRECISION predecimos la edad')
print('La derechohabiencia tiene efecto estadisticamente')
print('significativo en la edad de muerte')

COMPARATIVA MODELO 2 — EDAD DE MUERTE

ENFOQUE EXPLICATIVO (statsmodels OLS)
----------------------------------------
R2              : 0.0279
R2 ajustado     : 0.0279
AIC             : 423979.03
Efecto derechohabiencia: +3.46 anios de vida
Pregunta        : Cuantos anios mas vive quien tiene cobertura?

ENFOQUE PREDICTIVO (Random Forest)
----------------------------------------
R2              : 0.2696
RMSE            : 14.61 anios
Variable mas imp: Nivel Educativo (0.814)
Pregunta        : A que edad morira esta persona?

CONCLUSION
----------------------------------------
OLS: cuantos anios de vida APORTA cada variable
Random Forest: con que PRECISION predecimos la edad
La derechohabiencia tiene efecto estadisticamente
significativo en la edad de muerte


---
## 5. Comparativa Final — Ambos Modelos
---

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Comparativa Final — Predictivo vs Explicativo\nCenso INEGI 2000',
             fontsize=14, fontweight='bold')

colores_comp = ['#2196F3', '#4CAF50']

# Modelo 1
modelos1 = ['Explicativo\n(Logit)', 'Predictivo\n(Decision Tree)']
metricas_m1 = [resultado_logit.prsquared, accuracy]
bars1 = axes[0].bar(modelos1, metricas_m1, color=colores_comp, width=0.4)
axes[0].set_title('Modelo 1 — Derechohabiencia\nPseudo R2 vs Accuracy', fontsize=11)
axes[0].set_ylabel('Metrica')
axes[0].set_ylim(0, 1)
for bar, val in zip(bars1, metricas_m1):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontweight='bold')

# Modelo 2
modelos2 = ['Explicativo\n(OLS)', 'Predictivo\n(Random Forest)']
metricas_m2 = [resultado_ols.rsquared, r2_rf]
bars2 = axes[1].bar(modelos2, metricas_m2, color=colores_comp, width=0.4)
axes[1].set_title('Modelo 2 — Edad de Muerte\nR2 Comparativo', fontsize=11)
axes[1].set_ylabel('R2')
axes[1].set_ylim(0, max(metricas_m2) * 1.3)
for bar, val in zip(bars2, metricas_m2):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
ruta = FIGURES_DIR / 'comparativa_final.png'
plt.savefig(ruta, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {ruta}')

conn.close()
print('Conexion cerrada')

Guardado: /home/carpuro/mcd/censo-salud-mexico-2000/figures/comparativa_final.png
Conexion cerrada


## 6. Conclusiones

### Modelo 1 — Derechohabiencia

El **Logit** (explicativo) confirma que el nivel educativo y el tipo de localidad son los factores con mayor efecto en la probabilidad de tener cobertura, con significancia estadística. El sexo no resultó significativo al controlar por las demás variables — lo que sugiere que la brecha de género en el acceso a salud está mediada por educación y geografía, no es directa.

El **Decision Tree** (predictivo) alcanzó un accuracy de **71.45%** con solo 5 niveles de profundidad. La variable más importante fue Nivel Educativo (77.1% de la importancia), coincidiendo con el enfoque explicativo. El estado de residencia tuvo importancia cercana a cero — no porque no importe geográficamente, sino porque con `max_depth=5` el árbol no tiene espacio para explotar una variable categórica con 32 valores.

### Modelo 2 — Edad de Muerte

El **OLS** (explicativo) encontró que tener derechohabiencia está asociado a **+3.46 años de vida** (p < 0.0001), controlando por sexo, educación, localidad y estado. Este es el hallazgo más sólido del análisis: cuantifica con precisión estadística el impacto de la cobertura médica en la longevidad. El R² bajo (0.028) es esperado — la edad de muerte tiene alta varianza individual por causas no capturables con datos sociodemográficos.

El **Random Forest** (predictivo) logró R² = **0.2696** con RMSE de 14.61 años — diez veces más poder predictivo que el OLS (R² = 0.028). La diferencia confirma que las relaciones entre variables demográficas y edad de muerte son marcadamente no lineales, algo que el OLS estructuralmente no puede capturar.

### Comparativa General

| | Explicativo | Predictivo |
|---|---|---|
| Fortaleza | Interpretabilidad, p-valores | Mayor precisión, no linealidad |
| Limitación | Supuestos paramétricos, linealidad | Caja negra, no inferencia |
| Ideal para | Investigación, política pública | Sistemas de predicción |
| Pregunta | ¿Por qué y cuánto? | ¿Con qué precisión? |
| Modelo 1 | Pseudo R² = 0.12 | Accuracy = 71.45% |
| Modelo 2 | R² = 0.028, +3.46 años derechohabiencia | R² = 0.2696, RMSE = 14.6 años |

> **Conclusión:** Ambos enfoques son complementarios y no compiten entre sí. El explicativo cuantifica el *mecanismo*; el predictivo cuantifica la *precisión*. En investigación de política pública — que es el contexto de este análisis — el enfoque explicativo es el más relevante: no nos interesa predecir quién tendrá cobertura, sino entender qué factores estructurales la determinan para poder intervenirlos.